In [2]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import plotly.io as pio
pio.renderers.default = "notebook_connected"

DATA_PATH = r"C:\Users\mmari\Downloads\export.csv"

COL_PDL = "pdl_id"
COL_DT  = "datetime"
COL_PWR = "p_kw"

In [3]:
raw = pd.read_csv(DATA_PATH, sep=",")
raw = raw.rename(columns={"id": COL_PDL, "horodate": COL_DT, "valeur": COL_PWR})

# parsing datetime
raw[COL_DT] = pd.to_datetime(raw[COL_DT], utc=True, errors="coerce")

# option: forcer timezone Europe/Paris (recommandé pour "jour" et "hh_index")
# si raw[COL_DT] est tz-aware: tz_convert marche
# si tz-naive: tz_localize
if raw[COL_DT].dt.tz is None:
    raw[COL_DT] = raw[COL_DT].dt.tz_localize("Europe/Paris")
else:
    raw[COL_DT] = raw[COL_DT].dt.tz_convert("Europe/Paris")

df = raw.dropna(subset=[COL_PDL, COL_DT, COL_PWR]).copy()

df[COL_PWR] = pd.to_numeric(df[COL_PWR], errors="coerce")
df = df.dropna(subset=[COL_PWR])

df.head()


,pdl_id,datetime,p_kw
0,476866365062,2023-11-01 00:00:00+01:00,1328.1
1,476866365062,2023-11-01 00:30:00+01:00,240.7
2,476866365062,2023-11-01 01:00:00+01:00,245.7
3,476866365062,2023-11-01 01:30:00+01:00,324.5
4,476866365062,2023-11-01 02:00:00+01:00,238.0


In [4]:
print("n_rows:", len(df))
print("n_clients:", df[COL_PDL].nunique())
print(df.dtypes)

n_rows: 8736000
n_clients: 500
pdl_id                             int64
datetime    datetime64[ns, Europe/Paris]
p_kw                             float64
dtype: object


In [5]:
df["date"] = df[COL_DT].dt.date
df["dow"] = df[COL_DT].dt.dayofweek # day of week
df["is_weekend"] = df["dow"] >= 5
df["hh_index"] = ((df[COL_DT].dt.hour * 60) + df[COL_DT].dt.minute) // 30  # index de la demi-heure, de 0 à 47

In [6]:
daily = (df
    .assign(energy_kwh_step=df[COL_PWR] * 0.5)
    .groupby([COL_PDL, "date"], as_index=False)
    .agg(
        daily_kwh=("energy_kwh_step", "sum"),
        daily_mean_kw=(COL_PWR, "mean"),
        daily_max_kw=(COL_PWR, "max"),
        n_steps=(COL_PWR, "size"),
    )
)

daily.head()

,pdl_id,date,daily_kwh,daily_mean_kw,daily_max_kw,n_steps
0,1704875583,2023-11-01,13571.55,565.481250,3291.2,48
1,1704875583,2023-11-02,15846.95,660.289583,3381.8,48
2,1704875583,2023-11-03,14021.70,584.237500,3447.7,48
3,1704875583,2023-11-04,16563.30,690.137500,3630.5,48
4,1704875583,2023-11-05,14427.50,601.145833,3849.8,48


In [7]:
# seuil par client sur les jours > 0
def q20_positive(s: pd.Series):
    s = s[s > 0]
    if len(s) == 0:
        return np.nan
    return s.quantile(0.2)

daily["th_pdl"] = (daily.groupby(COL_PDL)["daily_kwh"]
                   .transform(q20_positive))

# fallback si un client n'a pas de jours >0 : on le marque jamais actif
daily["is_active_day"] = (daily["daily_kwh"] >= daily["th_pdl"]).fillna(False)

daily.head()

,pdl_id,date,daily_kwh,daily_mean_kw,daily_max_kw,n_steps,th_pdl,is_active_day
0,1704875583,2023-11-01,13571.55,565.481250,3291.2,48,5881.95,True
1,1704875583,2023-11-02,15846.95,660.289583,3381.8,48,5881.95,True
2,1704875583,2023-11-03,14021.70,584.237500,3447.7,48,5881.95,True
3,1704875583,2023-11-04,16563.30,690.137500,3630.5,48,5881.95,True
4,1704875583,2023-11-05,14427.50,601.145833,3849.8,48,5881.95,True


In [8]:
daily2 = daily.copy()
daily2["date_ts"] = pd.to_datetime(daily2["date"])
daily2["month"] = daily2["date_ts"].dt.month

def season_from_month(m):
    # à adapter si vous avez une définition métier différente
    if m in (12, 1, 2):
        return "winter"
    if m in (6, 7, 8):
        return "summer"
    return "mid"

daily2["season"] = daily2["month"].map(season_from_month)

season_stats = (daily2
    .groupby([COL_PDL, "season"], as_index=False)
    .agg(mean_daily_kwh=("daily_kwh", "mean"))
    .pivot(index=COL_PDL, columns="season", values="mean_daily_kwh")
    .reset_index()
)

# colonnes manquantes -> 0 (si un client n'a pas de points dans une saison)
for c in ["winter", "summer", "mid"]:
    if c not in season_stats.columns:
        season_stats[c] = 0.0

global_mean = (daily2.groupby(COL_PDL, as_index=False)
               .agg(mean_daily_kwh_global=("daily_kwh", "mean")))

season_stats = season_stats.merge(global_mean, on=COL_PDL, how="left", validate="one_to_one")

eps = 1e-9
season_stats["r_global"] = 1.0  # par définition (référence)
season_stats["r_mid"]    = season_stats["mid"]    / (season_stats["mean_daily_kwh_global"] + eps)
season_stats["r_summer"] = season_stats["summer"] / (season_stats["mean_daily_kwh_global"] + eps)
season_stats["r_winter"] = season_stats["winter"] / (season_stats["mean_daily_kwh_global"] + eps)

season_stats = season_stats[[COL_PDL, "r_global", "r_mid", "r_summer", "r_winter"]]    
season_stats.head()

,pdl_id,r_global,r_mid,r_summer,r_winter
0,1704875583,1.0,0.889323,0.414209,1.812366
1,6674572658,1.0,0.963229,0.450283,1.628896
2,9993623468,1.0,0.826638,0.770493,1.576848
3,10607320546,1.0,1.114088,0.858928,0.915701
4,11239534806,1.0,0.895518,0.330523,1.884650


In [9]:
activity = (daily
    .groupby(COL_PDL, as_index=False)
    .agg(
        n_days=(COL_PDL, "size"),
        n_active_days=("is_active_day", "sum"),
        active_day_rate=("is_active_day", "mean"),
        mean_daily_kwh=("daily_kwh", "mean"),
        p95_daily_kwh=("daily_kwh", lambda s: s.quantile(0.95)),
        cv_daily_kwh=("daily_kwh", lambda s: (s.std() / s.mean()) if s.mean() != 0 else np.nan),
    )
)

activity.head()

,pdl_id,n_days,n_active_days,active_day_rate,mean_daily_kwh,p95_daily_kwh,cv_daily_kwh
0,1704875583,364,291,0.799451,20274.808242,39500.6325,0.662061
1,6674572658,364,291,0.799451,19274.249725,44640.9575,0.731608
2,9993623468,364,281,0.771978,15160.003297,33961.7925,0.688953
3,10607320546,364,291,0.799451,17192.027885,34662.5475,0.436081
4,11239534806,364,291,0.799451,13282.210027,34550.7325,0.825156


In [10]:
def runs_and_gaps(active_series: pd.Series):
    # active_series: bool list in chronological order
    runs = []
    gaps = []
    run = 0
    gap = 0
    
    for v in active_series.astype(bool):
        if v:
            run += 1
            if gap > 0:
                gaps.append(gap)
                gap = 0
        else:
            gap += 1
            if run > 0:
                runs.append(run)
                run = 0

    if run > 0:
        runs.append(run)
    if gap > 0:
        gaps.append(gap)

    return pd.Series({
        "n_runs": len(runs),
        "mean_run_len": float(np.mean(runs)) if runs else 0.0,
        "max_run_len": float(np.max(runs)) if runs else 0.0,
        "mean_gap_len": float(np.mean(gaps)) if gaps else 0.0,
        "max_gap_len": float(np.max(gaps)) if gaps else 0.0,
    })

runs_stats = (daily
    .sort_values([COL_PDL, "date"])
    .groupby(COL_PDL)["is_active_day"]
    .apply(runs_and_gaps)
    .unstack()
    .reset_index()
)

runs_stats.head()

,pdl_id,n_runs,mean_run_len,max_run_len,mean_gap_len,max_gap_len
0,1704875583,16.0,18.187500,246.0,4.866667,23.0
1,6674572658,27.0,10.777778,140.0,2.703704,16.0
2,9993623468,14.0,20.071429,102.0,6.384615,55.0
3,10607320546,40.0,7.275000,67.0,1.871795,8.0
4,11239534806,10.0,29.100000,151.0,8.111111,38.0


In [11]:
daily_dt = pd.to_datetime(daily["date"])
daily["dow"] = daily_dt.dt.dayofweek
daily["is_weekend"] = daily["dow"] >= 5

week_pattern = (daily
    .groupby([COL_PDL, "is_weekend"], as_index=False)
    .agg(active_rate=("is_active_day", "mean"),
         mean_kwh=("daily_kwh", "mean"))
    .pivot(index=COL_PDL, columns="is_weekend")
)

week_pattern.columns = [f"{a}_{'weekend' if b else 'weekday'}" for a, b in week_pattern.columns]
week_pattern = week_pattern.reset_index()

week_pattern.head()

,pdl_id,active_rate_weekday,active_rate_weekend,mean_kwh_weekday,mean_kwh_weekend
0,1704875583,0.788462,0.826923,20020.331346,20911.000481
1,6674572658,0.796154,0.807692,19034.687692,19873.154808
2,9993623468,0.773077,0.769231,14652.154615,16429.625000
3,10607320546,0.796154,0.807692,16956.096731,17781.855769
4,11239534806,0.792308,0.817308,12734.230769,14652.158173


In [12]:
features_pdl = (activity
    .merge(runs_stats, on=COL_PDL, how="left", validate="one_to_one")
    .merge(week_pattern, on=COL_PDL, how="left", validate="one_to_one")
    .merge(season_stats, on=COL_PDL, how="left", validate="one_to_one")
)

assert features_pdl[COL_PDL].is_unique, "ERREUR: plus d'une ligne par client => un merge a explosé"
print("OK: 1 ligne par client:", len(features_pdl), "clients:", features_pdl[COL_PDL].nunique())

features_pdl.head()

OK: 1 ligne par client: 500 clients: 500


,pdl_id,n_days,n_active_days,active_day_rate,mean_daily_kwh,p95_daily_kwh,cv_daily_kwh,n_runs,mean_run_len,max_run_len,mean_gap_len,max_gap_len,active_rate_weekday,active_rate_weekend,mean_kwh_weekday,mean_kwh_weekend,r_global,r_mid,r_summer,r_winter
0,1704875583,364,291,0.799451,20274.808242,39500.6325,0.662061,16.0,18.187500,246.0,4.866667,23.0,0.788462,0.826923,20020.331346,20911.000481,1.0,0.889323,0.414209,1.812366
1,6674572658,364,291,0.799451,19274.249725,44640.9575,0.731608,27.0,10.777778,140.0,2.703704,16.0,0.796154,0.807692,19034.687692,19873.154808,1.0,0.963229,0.450283,1.628896
2,9993623468,364,281,0.771978,15160.003297,33961.7925,0.688953,14.0,20.071429,102.0,6.384615,55.0,0.773077,0.769231,14652.154615,16429.625000,1.0,0.826638,0.770493,1.576848
3,10607320546,364,291,0.799451,17192.027885,34662.5475,0.436081,40.0,7.275000,67.0,1.871795,8.0,0.796154,0.807692,16956.096731,17781.855769,1.0,1.114088,0.858928,0.915701
4,11239534806,364,291,0.799451,13282.210027,34550.7325,0.825156,10.0,29.100000,151.0,8.111111,38.0,0.792308,0.817308,12734.230769,14652.158173,1.0,0.895518,0.330523,1.884650


In [13]:
features_pdl["seasonality_amp"] = features_pdl[["r_mid","r_summer","r_winter"]].max(axis=1) - features_pdl[["r_mid","r_summer","r_winter"]].min(axis=1)
features_pdl["winter_minus_summer"] = features_pdl["r_winter"] - features_pdl["r_summer"]

In [14]:
feature_cols = [
    "active_day_rate", "n_runs", "mean_run_len", "max_run_len",
    "mean_gap_len", "max_gap_len",
    "mean_daily_kwh", "p95_daily_kwh", "cv_daily_kwh",
    "active_rate_weekday", "active_rate_weekend",
    "mean_kwh_weekday", "mean_kwh_weekend", "winter_minus_summer", 
    "seasonality_amp", "r_global",	"r_mid", "r_summer", "r_winter",
]

X = features_pdl[feature_cols].copy()
X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [15]:
scores = {}
for k in range(2, 7):
    km = KMeans(n_clusters=k, n_init=20, random_state=42)
    labels = km.fit_predict(X_scaled)
    scores[k] = silhouette_score(X_scaled, labels)

scores

{2: 0.22088447703346464,
 3: 0.24609800736700174,
 4: 0.25559658702052507,
 5: 0.1730877039450529,
 6: 0.17456543102137312}

In [16]:
best_k = max(scores, key=scores.get)
best_k

4

In [17]:
best_k = 10
kmeans = KMeans(n_clusters=best_k, n_init=50, random_state=42)
features_pdl["cluster"] = kmeans.fit_predict(X_scaled)
features_pdl[["pdl_id","cluster"]].head()

,pdl_id,cluster
0,1704875583,3
1,6674572658,3
2,9993623468,0
3,10607320546,5
4,11239534806,6


In [18]:
cluster_profile = features_pdl.groupby("cluster")[feature_cols].mean().sort_index()
cluster_profile

,active_day_rate,n_runs,mean_run_len,max_run_len,mean_gap_len,max_gap_len,mean_daily_kwh,p95_daily_kwh,cv_daily_kwh,active_rate_weekday,active_rate_weekend,mean_kwh_weekday,mean_kwh_weekend,winter_minus_summer,seasonality_amp,r_global,r_mid,r_summer,r_winter
cluster,,,,,,,,,,,,,,,,,,,
0,0.797239,21.754237,14.355307,153.237288,3.782461,22.084746,15216.287378,36506.048242,0.693038,0.792601,0.808833,14885.181514,16044.052037,1.000951,1.014122,1.0,0.922096,0.579312,1.580263
1,0.586996,16.333333,13.638184,101.555556,10.264629,85.777778,7719.409768,22634.228611,1.033604,0.585043,0.591880,7599.167714,8020.014904,-0.668870,1.282587,1.0,1.167569,1.166870,0.498000
2,0.770966,9.473684,33.224480,191.000000,10.945004,43.473684,11898.860382,27154.100789,0.785594,0.769231,0.775304,11660.337318,12495.168041,-0.664039,0.896737,1.0,0.951896,1.377784,0.713745
3,0.797825,20.521127,15.438872,187.084507,4.065563,22.309859,20284.381241,48239.809718,0.726290,0.791874,0.812703,19981.443017,21041.726801,1.404431,1.404692,1.0,0.876662,0.423612,1.828043
4,0.799451,2.333333,129.333333,262.000000,36.500000,65.333333,13979.979212,34992.039167,0.759981,0.800000,0.798077,13737.166282,14587.011538,-0.063412,0.534232,1.0,1.071118,0.961192,0.897780
5,0.798524,24.790698,12.693400,116.593023,3.289999,17.441860,17589.128466,37161.851047,0.561561,0.790877,0.817643,17198.610651,18565.423004,0.473914,0.590235,1.0,0.931053,0.832532,1.306446
6,0.790865,7.416667,42.833036,208.125000,12.640669,50.125000,15355.934364,36751.785937,0.764898,0.791026,0.790465,15083.157099,16037.877524,1.135338,1.165781,1.0,0.991501,0.443839,1.579177
7,0.686328,17.294118,16.029830,111.235294,7.754298,60.647059,12941.670726,37029.358088,0.921221,0.682805,0.695136,12754.618428,13409.301471,1.060530,1.118291,1.0,0.871226,0.599999,1.660529
8,0.797938,21.420290,14.981186,153.840580,3.987897,21.376812,16779.226636,51098.496630,0.925933,0.795819,0.803233,16443.697383,17618.049770,1.692276,1.708626,1.0,0.748852,0.406889,2.099165


In [19]:
import plotly.express as px
import plotly.graph_objects as go

def plot_client_daily(pdl_id, daily_df, n_days=200):
    sub = daily_df[daily_df[COL_PDL] == pdl_id].sort_values("date").tail(n_days).copy()
    sub["date_ts"] = pd.to_datetime(sub["date"])

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=sub["date_ts"], y=sub["daily_kwh"],
        mode="lines+markers", name="daily_kwh"
    ))

    active = sub[sub["is_active_day"]]
    fig.add_trace(go.Scatter(
        x=active["date_ts"], y=active["daily_kwh"],
        mode="markers", name="active_day"
    ))

    fig.update_layout(title=f"PDL {pdl_id} : daily_kwh + jours actifs (last {n_days} days)",
                      xaxis_title="date", yaxis_title="kWh/jour")
    fig.show()

for c in sorted(features_pdl["cluster"].unique()):
    sample = features_pdl[features_pdl["cluster"] == c].sample(min(1, (features_pdl["cluster"]==c).sum()), random_state=42)
    print("Cluster", c, "samples:", sample[COL_PDL].tolist())
    for pid in sample[COL_PDL].tolist():
        plot_client_daily(pid, daily, n_days=250)

Cluster 0 samples: [496816889201]


Cluster 1 samples: [711954092132]


Cluster 2 samples: [16277393756]


Cluster 3 samples: [389049510861]


Cluster 4 samples: [593948855549]


Cluster 5 samples: [885451189626]


Cluster 6 samples: [425577291923]


Cluster 7 samples: [63731311654]


Cluster 8 samples: [310552464401]


Cluster 9 samples: [891339182265]


In [20]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# --- prépare X ---
X = features_pdl[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
X_scaled = StandardScaler().fit_transform(X)

# --- PCA 2D ---
pca = PCA(n_components=2, random_state=42)
Z = pca.fit_transform(X_scaled)

# X = features_pdl[feature_cols] (avant scaling) est déjà un DataFrame avec les colonnes feature_cols

pca_df = pd.concat(
    [
        pd.DataFrame({"PC1": Z[:, 0], "PC2": Z[:, 1]}),
        features_pdl[[COL_PDL, "cluster"] + feature_cols].reset_index(drop=True)
    ],
    axis=1
)

pca_df["cluster"] = pca_df["cluster"].astype(str)
pca_df[COL_PDL] = pca_df[COL_PDL].astype(str)

fig = px.scatter(
    pca_df,
    x="PC1", y="PC2",
    color="cluster",
    hover_data=[COL_PDL] + feature_cols,
    title=(
        f"PCA (2D) — variance expliquée: "
        f"PC1={pca.explained_variance_ratio_[0]:.2%}, "
        f"PC2={pca.explained_variance_ratio_[1]:.2%}"
    )
)
fig.update_traces(marker=dict(size=7, opacity=0.75))
fig.show()

In [21]:
from sklearn.manifold import TSNE

# --- X + scaling (si tu as déjà X_scaled, tu peux sauter ça) ---
X = features_pdl[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
X_scaled = StandardScaler().fit_transform(X)

# --- t-SNE params safe ---
n = X_scaled.shape[0]
perp = int(min(50, max(5, (n - 1) // 3)))  # perplexity doit être < (n-1)/3 environ

tsne = TSNE(
    n_components=2,
    perplexity=perp,
    learning_rate="auto",
    init="pca",
    random_state=42
)

T = tsne.fit_transform(X_scaled)

# --- DataFrame pour plot: coords + cluster + pdl + features ---
tsne_df = pd.concat(
    [
        pd.DataFrame({"tSNE1": T[:, 0], "tSNE2": T[:, 1]}),
        features_pdl[[COL_PDL, "cluster"] + feature_cols].reset_index(drop=True)
    ],
    axis=1
)

tsne_df["cluster"] = tsne_df["cluster"].astype(str)   # légende DISCRÈTE
tsne_df[COL_PDL] = tsne_df[COL_PDL].astype(str)

# --- plot ---
fig = px.scatter(
    tsne_df,
    x="tSNE1", y="tSNE2",
    color="cluster",
    hover_data=[COL_PDL] + feature_cols,
    title=f"t-SNE (2D) — perplexity={perp}"
)
fig.update_traces(marker=dict(size=7, opacity=0.75))
fig.show()

In [22]:
# rattacher le cluster à daily
daily_cluster = daily.merge(
    features_pdl[[COL_PDL, "cluster"]],
    on=COL_PDL,
    how="left",
    validate="many_to_one"
)

assert daily_cluster["cluster"].notna().all()

In [24]:
mean_daily_cluster = (
    daily_cluster
    .groupby(["cluster", "date"], as_index=False)
    .agg(mean_daily_kwh=("daily_kwh", "mean"))
)

In [25]:
fig = px.line(
    mean_daily_cluster,
    x="date",
    y="mean_daily_kwh",
    color=mean_daily_cluster["cluster"].astype(str),
    title="Courbe moyenne journalière par cluster",
    labels={"mean_daily_kwh": "kWh / jour", "cluster": "cluster"}
)
fig.show()


In [26]:
df_cluster = df.merge(
    features_pdl[[COL_PDL, "cluster"]],
    on=COL_PDL,
    how="left",
    validate="many_to_one"
)

assert df_cluster["cluster"].notna().all()

In [27]:
mean_intraday_cluster = (
    df_cluster
    .groupby(["cluster", "hh_index"], as_index=False)
    .agg(mean_kw=(COL_PWR, "mean"))
)

mean_intraday_cluster["cluster"] = mean_intraday_cluster["cluster"].astype(str)

In [28]:
fig = px.line(
    mean_intraday_cluster,
    x="hh_index",
    y="mean_kw",
    color="cluster",
    title="Profil intrajournalier moyen par cluster (30 min)",
    labels={
        "hh_index": "Demi-heure (0 = 00:00)",
        "mean_kw": "Puissance moyenne (kW)",
        "cluster": "cluster"
    }
)

fig.update_layout(
    xaxis=dict(
        tickmode="array",
        tickvals=list(range(0, 48, 4)),
        ticktext=[f"{h:02d}:00" for h in range(0, 24, 2)]
    )
)

fig.show()

In [29]:
df_cluster["is_weekend"] = df_cluster[COL_DT].dt.dayofweek >= 5

mean_intraday_wk = (
    df_cluster
    .groupby(["cluster", "is_weekend", "hh_index"], as_index=False)
    .agg(mean_kw=(COL_PWR, "mean"))
)

mean_intraday_wk["cluster"] = mean_intraday_wk["cluster"].astype(str)
mean_intraday_wk["day_type"] = mean_intraday_wk["is_weekend"].map({True: "weekend", False: "weekday"})

In [30]:
fig = px.line(
    mean_intraday_wk,
    x="hh_index",
    y="mean_kw",
    color="cluster",
    facet_col="day_type",
    title="Profil intrajournalier moyen par cluster — semaine vs week-end"
)

fig.update_layout(
    xaxis=dict(
        tickmode="array",
        tickvals=list(range(0, 48, 4)),
        ticktext=[f"{h:02d}:00" for h in range(0, 24, 2)]
    )
)

fig.show()

In [31]:
summary = (features_pdl
    .groupby("cluster")
    .agg(
        n_clients=(COL_PDL, "size"),
        active_day_rate=("active_day_rate", "mean"),
        max_gap_len=("max_gap_len", "mean"),
        winter_minus_summer=("winter_minus_summer", "mean"),
        cv_daily_kwh=("cv_daily_kwh", "mean"),
        r_summer=("r_summer", "mean")
    )
)
summary

,n_clients,active_day_rate,max_gap_len,winter_minus_summer,cv_daily_kwh,r_summer
cluster,,,,,,
0,118,0.797239,22.084746,1.000951,0.693038,0.579312
1,9,0.586996,85.777778,-0.668870,1.033604,1.166870
2,19,0.770966,43.473684,-0.664039,0.785594,1.377784
3,71,0.797825,22.309859,1.404431,0.726290,0.423612
4,3,0.799451,65.333333,-0.063412,0.759981,0.961192
5,86,0.798524,17.441860,0.473914,0.561561,0.832532
6,24,0.790865,50.125000,1.135338,0.764898,0.443839
7,17,0.686328,60.647059,1.060530,0.921221,0.599999
8,69,0.797938,21.376812,1.692276,0.925933,0.406889


In [33]:
# --- 1. Définition des étiquettes binaires (0 pour RP, 1 sinon) ---
# Basé sur notre interprétation des clusters
# RP : 0, 3, 5, 8, 9
# RS/Atypique : 1, 2, 4, 6, 7
cluster_to_label = {
    0: 0,
    3: 0,
    5: 0,
    8: 0,
    9: 0,
    1: 1,
    2: 1,
    4: 1,
    6: 1,
    7: 1
}

# --- 2. Création de la table de labels ---
# On part du dataframe `features_pdl` qui contient le `pdl_id` et le `cluster`
labels_df = features_pdl[[COL_PDL, "cluster"]].copy()

# On utilise la méthode .map() pour créer la colonne "label"
labels_df["label"] = labels_df["cluster"].map(cluster_to_label)

# On renomme la colonne 'pdl_id' en 'id' pour le fichier de sortie
output_df = labels_df[[COL_PDL, "label", "cluster"]].rename(columns={COL_PDL: "id"})


# --- 3. Sauvegarde dans un fichier CSV ---
output_path = r"C:\Users\mmari\Desktop\2A GI ENPC\S2\Data et energie\output.xlsx"
output_df.to_csv(output_path, index=False, sep=",")

print(f"Le fichier de labels a été sauvegardé avec succès ici : {output_path}")
print("\nAperçu des 5 premières lignes :")
output_df.head()

Le fichier de labels a été sauvegardé avec succès ici : C:\Users\mmari\Desktop\2A GI ENPC\S2\Data et energie\output.xlsx

Aperçu des 5 premières lignes :


,id,label,cluster
0,1704875583,0,3
1,6674572658,0,3
2,9993623468,0,0
3,10607320546,0,5
4,11239534806,1,6
